In [0]:
# Read parcel_metadata.csv (added file: prefix)
metadata_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("file:/Workspace/Users/srinath.mahto@gmail.com/Carnot assignment/parcel_metadata.csv")

# Read parcel_readings.csv (added file: prefix)
readings_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("file:/Workspace/Users/srinath.mahto@gmail.com/Carnot assignment/parcel_readings.csv")

# Display the dataframes to view the actual data grids
display(metadata_df)
display(readings_df.limit(20))

parcel_id,mill_id,crop_type,sowing_date,area_hectares
PARCEL_001,MILL_NORTH,sugarcane,2026-02-10,9.03
PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97
PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35
PARCEL_004,MILL_NORTH,sugarcane,2026-01-02,3.18
PARCEL_005,MILL_NORTH,soybean,2026-02-08,2.79
PARCEL_006,MILL_WEST,sugarcane,2026-02-11,5.67
PARCEL_007,MILL_NORTH,sugarcane,2026-01-18,8.53
PARCEL_008,MILL_EAST,sugarcane,2026-01-22,2.98
PARCEL_009,MILL_EAST,sugarcane,2026-02-18,1.57
PARCEL_010,MILL_EAST,sugarcane,2026-01-07,7.44


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
PARCEL_018,16/05/2026,0.595,15.4,0.0,error
PARCEL_014,2026-01-27,0.457,27.6,0.0,OK
PARCEL_006,2026-05-17,0.497,25.8,0.0,OK
PARCEL_004,10/02/2026,0.81,25.0,0.0,OK
PARCEL_002,2026-01-17,0.269,33.2,0.0,OK
PARCEL_015,2026-03-19,0.873,26.3,0.0,OK
PARCEL_020,2026-05-08,0.806,16.8,2.7,OK
PARCEL_010,20-Jan-2026,0.315,27.9,0.9,OK
PARCEL_012,2026-05-09,0.894,23.8,0.0,NA
PARCEL_008,2026-02-07,0.278,15.7,4.1,error


DATA QUALITY AUDIT

### Audit 1 : Dataset Size

In [0]:
print("Metadata rows:", metadata_df.count())
print("Readings rows:", readings_df.count())

Metadata rows: 28
Readings rows: 3447


In [0]:
metadata_df.printSchema()

readings_df.printSchema()

root
 |-- parcel_id: string (nullable = true)
 |-- mill_id: string (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- sowing_date: date (nullable = true)
 |-- area_hectares: double (nullable = true)

root
 |-- parcel_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- ndvi_value: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- rainfall_mm: double (nullable = true)
 |-- sensor_status: string (nullable = true)



Note:- The reading date column was loaded as string and will be converted to date during transformation.

In [0]:
readings_df.select("date").distinct().show(50, False)

+-----------+
|date       |
+-----------+
|2026-02-01 |
|07/01/2026 |
|2026-01-30 |
|26/05/2026 |
|30-Jan-2026|
|2026-01-21 |
|05-Feb-2026|
|2026-05-21 |
|2026-04-05 |
|2026-03-14 |
|13-Mar-2026|
|09-Apr-2026|
|2026-03-03 |
|19/04/2026 |
|30/05/2026 |
|24/03/2026 |
|26/01/2026 |
|29-Apr-2026|
|02-May-2026|
|2026-02-07 |
|11-Mar-2026|
|09/04/2026 |
|2026-04-17 |
|2026-05-10 |
|17/02/2026 |
|2026-03-10 |
|28/04/2026 |
|27/01/2026 |
|2026-01-06 |
|2026-03-28 |
|2026-03-21 |
|03-May-2026|
|08/05/2026 |
|05-Mar-2026|
|26/02/2026 |
|13/04/2026 |
|20-Mar-2026|
|2026-04-25 |
|16/01/2026 |
|30-Apr-2026|
|23-May-2026|
|28-Mar-2026|
|28/05/2026 |
|11/05/2026 |
|20-Jan-2026|
|2026-01-16 |
|2026-04-24 |
|18-Mar-2026|
|2026-02-16 |
|2026-02-28 |
+-----------+
only showing top 50 rows


In [0]:
from pyspark.sql.functions import col

readings_df.filter(
    col("date").contains("/")
).show(20, False)

+----------+----------+----------+-------------+-----------+-------------+
|parcel_id |date      |ndvi_value|temperature_c|rainfall_mm|sensor_status|
+----------+----------+----------+-------------+-----------+-------------+
|PARCEL_018|16/05/2026|0.595     |15.4         |0.0        |error        |
|PARCEL_004|10/02/2026|0.81      |25.0         |0.0        |OK           |
|PARCEL_010|10/05/2026|0.403     |18.7         |0.0        |OK           |
|PARCEL_020|18/04/2026|0.699     |35.0         |0.0        |OK           |
|PARCEL_024|11/03/2026|0.569     |24.2         |0.0        |OK           |
|PARCEL_023|11/02/2026|0.283     |15.3         |0.0        |OK           |
|PARCEL_009|08/04/2026|0.738     |25.7         |0.0        |OK           |
|PARCEL_007|27/05/2026|0.534     |17.3         |4.1        |NULL         |
|PARCEL_007|17/05/2026|0.71      |30.9         |0.0        |OK           |
|PARCEL_002|20/03/2026|0.757     |30.6         |0.0        |OK           |
|PARCEL_018|16/02/2026|0.

In [0]:
from pyspark.sql.functions import col

slash_dates = readings_df.filter(
    col("date").contains("/")
).count()

dash_dates = readings_df.filter(
    col("date").contains("-")
).count()

print("Slash dates:", slash_dates)
print("Dash dates:", dash_dates)

Slash dates: 686
Dash dates: 2761


Based on the data quality assessment, a significant issue was identified in the date column, where a mix of date formats ($dd-MM-yyyy$ and $dd/MM/yyyy$) is currently present. This discrepancy affects **686 rows**, representing approximately **20%** of the dataset.

To resolve this inconsistency, the recommended action is to standardize the format and convert the column to a consistent `DateType`. Taking this step is critical, as uniform formatting is required for reliable time-series analysis, accurate chronological sorting, and downstream date-based calculations.

In [0]:
from pyspark.sql.functions import *

audit_dates = readings_df.withColumn(
    "date_format",
    when(col("date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
    .when(col("date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
    .when(col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
    .otherwise("unknown")
)

audit_dates.groupBy("date_format").count().show()

+-----------+-----+
|date_format|count|
+-----------+-----+
| yyyy-MM-dd| 2431|
| dd/MM/yyyy|  686|
|    unknown|  330|
+-----------+-----+



In [0]:
from pyspark.sql.functions import *

audit_dates = readings_df.withColumn(
    "date_format",
    when(col("date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
    .when(col("date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
    .when(col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
    .otherwise("unknown")
)

audit_dates.filter(
    col("date_format") == "unknown"
).select("date").distinct().show(100, False)

+-----------+
|date       |
+-----------+
|30-Jan-2026|
|05-Feb-2026|
|13-Mar-2026|
|09-Apr-2026|
|29-Apr-2026|
|02-May-2026|
|11-Mar-2026|
|03-May-2026|
|05-Mar-2026|
|20-Mar-2026|
|30-Apr-2026|
|23-May-2026|
|28-Mar-2026|
|20-Jan-2026|
|18-Mar-2026|
|31-Mar-2026|
|23-Mar-2026|
|28-Apr-2026|
|06-May-2026|
|16-May-2026|
|02-Jan-2026|
|23-Feb-2026|
|18-May-2026|
|24-Jan-2026|
|01-May-2026|
|01-Feb-2026|
|25-Feb-2026|
|04-Jan-2026|
|25-May-2026|
|27-Mar-2026|
|12-Feb-2026|
|08-Mar-2026|
|13-Feb-2026|
|27-Jan-2026|
|13-May-2026|
|01-Mar-2026|
|06-Mar-2026|
|21-Apr-2026|
|10-Apr-2026|
|27-May-2026|
|22-Jan-2026|
|21-Jan-2026|
|09-Mar-2026|
|28-Feb-2026|
|09-Jan-2026|
|18-Apr-2026|
|17-May-2026|
|23-Jan-2026|
|07-Feb-2026|
|24-May-2026|
|16-Mar-2026|
|07-Apr-2026|
|22-May-2026|
|11-Jan-2026|
|06-Apr-2026|
|30-May-2026|
|26-Mar-2026|
|12-Mar-2026|
|25-Jan-2026|
|26-Apr-2026|
|06-Feb-2026|
|19-Jan-2026|
|14-Feb-2026|
|12-Jan-2026|
|20-Feb-2026|
|01-Jan-2026|
|27-Apr-2026|
|26-May-2026|
|25-Ap


A major data quality issue was identified involving **multiple date formats** within the date column, impacting a significant portion of the dataset across 3,447 rows. To ensure data integrity, the decision has been made to standardize the entire column into a single, uniform `DateType`. Resolving this inconsistency is a critical prerequisite, as clean and uniform date data is strictly required for executing reliable date calculations, accurate chronological sorting, and robust time-series analysis downstream.

In [0]:
#verifying again

from pyspark.sql.functions import *

audit_dates = readings_df.withColumn(
    "date_format",
    when(col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
    .when(col("date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
    .when(col("date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
    .otherwise("unknown")
)

audit_dates.groupBy("date_format").count().show()

+-----------+-----+
|date_format|count|
+-----------+-----+
| yyyy-MM-dd| 2431|
| dd/MM/yyyy|  686|
|dd-MMM-yyyy|  330|
+-----------+-----+



DATE AUDIT COMPLETE

Next Audit: NDVI Column

In [0]:
from pyspark.sql.functions import *

readings_df.select(
    min("ndvi_value").alias("min_ndvi"),
    max("ndvi_value").alias("max_ndvi"),
    avg("ndvi_value").alias("avg_ndvi")
).show()

+--------+--------+-------------------+
|min_ndvi|max_ndvi|           avg_ndvi|
+--------+--------+-------------------+
|  -1.976|   1.997|0.46423469683783114|
+--------+--------+-------------------+



In [0]:
from pyspark.sql.functions import col

invalid_ndvi = readings_df.filter(
    (col("ndvi_value") < -1) |
    (col("ndvi_value") > 1)
)

print(
    "Invalid NDVI Count:",
    invalid_ndvi.count()
)

Invalid NDVI Count: 104


In [0]:
display(
    invalid_ndvi.select(
        "parcel_id",
        "date",
        "ndvi_value"
    )
)

parcel_id,date,ndvi_value
PARCEL_017,2026-02-01,1.832
PARCEL_001,2026-01-22,1.817
PARCEL_013,2026-02-19,1.665
PARCEL_024,2026-04-19,1.626
PARCEL_015,10/05/2026,1.301
PARCEL_001,08/01/2026,1.708
PARCEL_024,2026-05-07,1.337
PARCEL_012,20-Apr-2026,-1.672
PARCEL_005,27/05/2026,-1.024
PARCEL_011,2026-05-24,1.261


In [0]:
#Check Missing NDVI

readings_df.filter(
    col("ndvi_value").isNull()
).count()

0

NVDI aduit complete

Aditing sensor_status now

In [0]:
readings_df.groupBy(
    "sensor_status"
).count().orderBy(
    col("count").desc()
).show(100, False)

+-------------+-----+
|sensor_status|count|
+-------------+-----+
|OK           |2890 |
|Error        |90   |
|ERROR        |80   |
|error        |76   |
|ok           |64   |
|OK           |57   |
|NA           |53   |
| OK          |53   |
|NULL         |43   |
|NaN          |41   |
+-------------+-----+




Based on the data profiling results, two primary issues were identified regarding sensor status inconsistencies:

* **Inconsistent Sensor Status Values:** A total of **3,253 rows** exhibit inconsistent casing and formatting for standard status labels (such as *Ok*, *OK*, *ok*, *Error*, *ERROR*, and *error*). Because the same underlying business meanings are represented differently, the recommended action is to standardize these values into a single, uniform casing convention. This eliminates duplication and ensures clean categorization for downstream reporting.
* **Ambiguous Sensor Statuses:** There are **137 rows** containing ambiguous or missing status values, specifically *NA*, *NULL*, and *NaN*. Since the true health of the sensor is unknown in these instances, the readings cannot be trusted for analytical purposes. Consequently, these records will be treated as bad sensor readings to prevent them from skewing data quality and operational insights.


In [0]:
#final check

readings_df.filter(
    col("sensor_status").isNull()
).count()

43



* **Inconsistent Sensor Status Casing:** A total of **3,253 rows** exhibit inconsistent casing variations for identical states (such as mixing uppercase, lowercase, and sentence case). Because these multiple values represent the exact same business meaning, the action is to standardize the column to a uniform casing convention to ensure accurate grouping and aggregation.
* **Unknown Sensor Status Values:** There are **180 rows** containing ambiguous or missing indicators, including *NA*, *NULL*, *NaN*, and actual structural nulls. Since the sensor health cannot be verified in these instances, these records will be formally mapped to a `bad` status string to prevent unverified data from corrupting downstream operational metrics.

Sensor_audit complete

Missing Values Audit for all columns.

In [0]:
from pyspark.sql.functions import *

metadata_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in metadata_df.columns
]).show()

readings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in readings_df.columns
]).show()

+---------+-------+---------+-----------+-------------+
|parcel_id|mill_id|crop_type|sowing_date|area_hectares|
+---------+-------+---------+-----------+-------------+
|        0|      0|        0|          0|            0|
+---------+-------+---------+-----------+-------------+

+---------+----+----------+-------------+-----------+-------------+
|parcel_id|date|ndvi_value|temperature_c|rainfall_mm|sensor_status|
+---------+----+----------+-------------+-----------+-------------+
|        0|   0|         0|            0|          0|           43|
+---------+----+----------+-------------+-----------+-------------+



No missing values found in parcel_metadata.csv

AUDIT SUMMARY SO FAR




* **Dates:** Standardizing the mixed formats into a single `DateType` so our time-series analysis actually works.
* **NDVI:** Dropping **104 records** that fall outside the valid $[-1, 1]$ range, since those numbers are physically impossible.
* **Sensor Status:** Cleaning up a mix of messy casing and missing data. We're simplifying everything into just two clear categories: valid entries become `good`, while errors, typos, and the **43 missing values** get marked as `bad`.

Next Audit: Duplicate Records

In [0]:
metadata_total = metadata_df.count()
metadata_distinct = metadata_df.dropDuplicates().count()

print("Metadata Total:", metadata_total)
print("Metadata Distinct:", metadata_distinct)

readings_total = readings_df.count()
readings_distinct = readings_df.dropDuplicates().count()

print("Readings Total:", readings_total)
print("Readings Distinct:", readings_distinct)

Metadata Total: 28
Metadata Distinct: 28
Readings Total: 3447
Readings Distinct: 3447



### Audit Conclusion

* **Metadata & Readings:** The duplicate check came back completely clean—**no duplicate rows** were found in either the metadata or the readings datasets. As a result, no deduplication action is required for these files.

Final Audit:- Join qualtiy

In [0]:
orphan_parcels = (
    readings_df.join(
        metadata_df,
        "parcel_id",
        "left_anti"
    )
)

print(
    "Orphan Parcel Count:",
    orphan_parcels.count()
)

Orphan Parcel Count: 40


In [0]:
orphan_parcels.select("parcel_id").distinct().show(100, False)

orphan_parcels.groupBy("parcel_id").count().show(100, False)

+----------+
|parcel_id |
+----------+
|PARCEL_098|
|PARCEL_099|
+----------+

+----------+-----+
|parcel_id |count|
+----------+-----+
|PARCEL_098|20   |
|PARCEL_099|20   |
+----------+-----+





### Metadata Match Issue

* **Missing Parcel IDs:** There are **40 rows** in the readings data that lack corresponding Parcel IDs in the metadata. Because this critical metadata is unavailable for enrichment, these records cannot be used for downstream analysis and will be excluded from the final joined dataset.



---

### Final Audit Findings Summary

* **Mixed Date Formats:** Standardizing **3,447 rows** into a uniform `DateType` for reliable time-series calculations.
* **Invalid NDVI Values:** Dropping **104 rows** with values outside the valid $[-1, 1]$ range.
* **Sensor Status Casing:** Standardizing **3,253 rows** with inconsistent casing to a single, clean format.
* **Unknown Sensor Statuses:** Mapping **180 rows** of *NA, NULL, and NaN* strings (including structural nulls) to `bad`.
* **Missing Sensor Status:** Marking **43 rows** with actual missing values as `bad` since their health can't be verified.
* **Duplicate Records:** **0 duplicates** found; no action required.
* **Orphan Parcel Records:** Excluding **40 rows** from the final join because they lack matching metadata.

In [0]:
#creating working copies
clean_readings = readings_df
clean_metadata = metadata_df


In [0]:
#standardize date column

from pyspark.sql.functions import *

clean_readings = clean_readings.withColumn(
    "parsed_date",
    when(
        col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        to_date(col("date"), "yyyy-MM-dd")
    )
    .when(
        col("date").rlike(r"^\d{2}/\d{2}/\d{4}$"),
        to_date(col("date"), "dd/MM/yyyy")
    )
    .when(
        col("date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"),
        to_date(col("date"), "dd-MMM-yyyy")
    )
)

In [0]:
#verifying

clean_readings.select(
    "date",
    "parsed_date"
).show(20, False)

+-----------+-----------+
|date       |parsed_date|
+-----------+-----------+
|16/05/2026 |2026-05-16 |
|2026-01-27 |2026-01-27 |
|2026-05-17 |2026-05-17 |
|10/02/2026 |2026-02-10 |
|2026-01-17 |2026-01-17 |
|2026-03-19 |2026-03-19 |
|2026-05-08 |2026-05-08 |
|20-Jan-2026|2026-01-20 |
|2026-05-09 |2026-05-09 |
|2026-02-07 |2026-02-07 |
|2026-01-10 |2026-01-10 |
|10/05/2026 |2026-05-10 |
|2026-01-16 |2026-01-16 |
|2026-03-24 |2026-03-24 |
|2026-03-12 |2026-03-12 |
|2026-01-01 |2026-01-01 |
|2026-02-04 |2026-02-04 |
|2026-01-26 |2026-01-26 |
|2026-01-28 |2026-01-28 |
|2026-04-10 |2026-04-10 |
+-----------+-----------+
only showing top 20 rows


In [0]:
#checking for failed conversions

clean_readings.filter(
    col("parsed_date").isNull()
).count()

0

In [0]:
clean_readings = (
    clean_readings
    .drop("date")
    .withColumnRenamed(
        "parsed_date",
        "date"
    )
)

In [0]:
clean_readings.printSchema()

root
 |-- parcel_id: string (nullable = true)
 |-- ndvi_value: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- rainfall_mm: double (nullable = true)
 |-- sensor_status: string (nullable = true)
 |-- date: date (nullable = true)



In [0]:
#cleaning NVCI

#Verifying

print(
    "Current Row Count:",
    clean_readings.count()
)

Current Row Count: 3447


In [0]:
from pyspark.sql.functions import col

clean_readings = clean_readings.filter(
    (col("ndvi_value") >= -1) &
    (col("ndvi_value") <= 1)
)

In [0]:
print(
    "Row Count After NDVI Cleaning:",
    clean_readings.count()
)

Row Count After NDVI Cleaning: 3343


In [0]:
#Verifying

clean_readings.filter(
    (col("ndvi_value") < -1) |
    (col("ndvi_value") > 1)
).count()

0

I identified 104 NDVI readings outside the valid range specified in the assignment. Since these values are physically invalid and could not be reliably repaired, I excluded them from the cleaned dataset.

### Cleaning sensor_status

In [0]:
from pyspark.sql.functions import *

clean_readings = clean_readings.withColumn(
    "sensor_status_clean",
    when(
        lower(col("sensor_status")) == "ok",
        "good"
    )
    .when(
        lower(col("sensor_status")) == "error",
        "bad"
    )
    .when(
        lower(col("sensor_status")).isin(
            "na",
            "null",
            "nan"
        ),
        "bad"
    )
    .when(
        col("sensor_status").isNull(),
        "bad"
    )
    .otherwise("bad")
)

In [0]:
clean_readings.groupBy(
    "sensor_status_clean"
).count().show()

+-------------------+-----+
|sensor_status_clean|count|
+-------------------+-----+
|               good| 2954|
|                bad|  389|
+-------------------+-----+



In [0]:
#counting before join

print(
    "Clean Readings:",
    clean_readings.count()
)

print(
    "Metadata:",
    clean_metadata.count()
)

Clean Readings: 3343
Metadata: 28


In [0]:
#performing inner join

cleaned_df = (
    clean_readings.join(
        clean_metadata,
        on="parcel_id",
        how="inner"
    )
)

print(
    cleaned_df.show()
)

+----------+----------+-------------+-----------+-------------+----------+-------------------+----------+---------+-----------+-------------+
| parcel_id|ndvi_value|temperature_c|rainfall_mm|sensor_status|      date|sensor_status_clean|   mill_id|crop_type|sowing_date|area_hectares|
+----------+----------+-------------+-----------+-------------+----------+-------------------+----------+---------+-----------+-------------+
|PARCEL_018|     0.595|         15.4|        0.0|        error|2026-05-16|                bad|MILL_SOUTH|sugarcane| 2026-01-11|         5.82|
|PARCEL_014|     0.457|         27.6|        0.0|           OK|2026-01-27|               good|MILL_NORTH|sugarcane| 2026-01-05|         9.39|
|PARCEL_006|     0.497|         25.8|        0.0|           OK|2026-05-17|               good| MILL_WEST|sugarcane| 2026-02-11|         5.67|
|PARCEL_004|      0.81|         25.0|        0.0|           OK|2026-02-10|               good|MILL_NORTH|sugarcane| 2026-01-02|         3.18|
|PARCE

In [0]:
print(
    "Rows After Join:",
    cleaned_df.count()
)

Rows After Join: 3303


In [0]:
#verifying no orphans remain

orphans_after_join = (
    clean_readings.join(
        clean_metadata,
        "parcel_id",
        "left_anti"
    )
)

print(
    "Remaining Orphans:",
    orphans_after_join.count()
)

Remaining Orphans: 40


In [0]:
#checking final schema

cleaned_df.printSchema()

root
 |-- parcel_id: string (nullable = true)
 |-- ndvi_value: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- rainfall_mm: double (nullable = true)
 |-- sensor_status: string (nullable = true)
 |-- date: date (nullable = true)
 |-- sensor_status_clean: string (nullable = false)
 |-- mill_id: string (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- sowing_date: date (nullable = true)
 |-- area_hectares: double (nullable = true)



In [0]:
cleaned_df.count()




3303

- Valid dates
- Valid NDVI values
- Standardized sensor status
- Valid metadata
- No orphan parcels


Analysis

For each crop_type:

Mean NDVI in 30 days BEFORE sowing
Mean NDVI in 30 days AFTER sowing
Ignore bad sensors
Return:


In [0]:
#creating

analysis_df = cleaned_df.filter(
    col("sensor_status_clean") == "good"
)

In [0]:
#verifying

analysis_df.groupBy(
    "sensor_status_clean"
).count().show()

+-------------------+-----+
|sensor_status_clean|count|
+-------------------+-----+
|               good| 2914|
+-------------------+-----+



In [0]:
analysis_df.select(
    "parcel_id",
    "crop_type",
    "date",
    "sowing_date"
).show(10, False)

+----------+---------+----------+-----------+
|parcel_id |crop_type|date      |sowing_date|
+----------+---------+----------+-----------+
|PARCEL_014|sugarcane|2026-01-27|2026-01-05 |
|PARCEL_006|sugarcane|2026-05-17|2026-02-11 |
|PARCEL_004|sugarcane|2026-02-10|2026-01-02 |
|PARCEL_002|sugarcane|2026-01-17|2026-01-16 |
|PARCEL_015|sugarcane|2026-03-19|2026-01-06 |
|PARCEL_020|sugarcane|2026-05-08|2026-02-19 |
|PARCEL_010|sugarcane|2026-01-20|2026-01-07 |
|PARCEL_019|sugarcane|2026-01-10|2026-01-18 |
|PARCEL_010|sugarcane|2026-05-10|2026-01-07 |
|PARCEL_001|sugarcane|2026-01-16|2026-02-10 |
+----------+---------+----------+-----------+
only showing top 10 rows


In [0]:
#calculating days from sowing

from pyspark.sql.functions import *

analysis_df = analysis_df.withColumn(
    "days_from_sowing",
    datediff(
        col("date"),
        col("sowing_date")
    )
)

In [0]:
analysis_df.select(
    "parcel_id",
    "date",
    "sowing_date",
    "days_from_sowing"
).show(20, False)

+----------+----------+-----------+----------------+
|parcel_id |date      |sowing_date|days_from_sowing|
+----------+----------+-----------+----------------+
|PARCEL_014|2026-01-27|2026-01-05 |22              |
|PARCEL_006|2026-05-17|2026-02-11 |95              |
|PARCEL_004|2026-02-10|2026-01-02 |39              |
|PARCEL_002|2026-01-17|2026-01-16 |1               |
|PARCEL_015|2026-03-19|2026-01-06 |72              |
|PARCEL_020|2026-05-08|2026-02-19 |78              |
|PARCEL_010|2026-01-20|2026-01-07 |13              |
|PARCEL_019|2026-01-10|2026-01-18 |-8              |
|PARCEL_010|2026-05-10|2026-01-07 |123             |
|PARCEL_001|2026-01-16|2026-02-10 |-25             |
|PARCEL_023|2026-03-24|2026-01-26 |57              |
|PARCEL_020|2026-03-12|2026-02-19 |21              |
|PARCEL_013|2026-01-01|2026-02-23 |-53             |
|PARCEL_025|2026-02-04|2026-02-07 |-3              |
|PARCEL_001|2026-01-26|2026-02-10 |-15             |
|PARCEL_013|2026-01-28|2026-02-23 |-26        

In [0]:
#creating before window

before_df = analysis_df.filter(
    (col("days_from_sowing") >= -30) &
    (col("days_from_sowing") < 0)
)

In [0]:
#creating after window

after_df = analysis_df.filter(
    (col("days_from_sowing") >= 0) &
    (col("days_from_sowing") <= 30)
)



In [0]:
#sanity check

print("Before Records:", before_df.count())
print("After Records :", after_df.count())



Before Records: 393
After Records : 594


In [0]:
analysis_df.select(
    "parcel_id",
    "date",
    "sowing_date",
    "days_from_sowing"
).show(20, False)

print("Before Records:", before_df.count())
print("After Records :", after_df.count())

+----------+----------+-----------+----------------+
|parcel_id |date      |sowing_date|days_from_sowing|
+----------+----------+-----------+----------------+
|PARCEL_014|2026-01-27|2026-01-05 |22              |
|PARCEL_006|2026-05-17|2026-02-11 |95              |
|PARCEL_004|2026-02-10|2026-01-02 |39              |
|PARCEL_002|2026-01-17|2026-01-16 |1               |
|PARCEL_015|2026-03-19|2026-01-06 |72              |
|PARCEL_020|2026-05-08|2026-02-19 |78              |
|PARCEL_010|2026-01-20|2026-01-07 |13              |
|PARCEL_019|2026-01-10|2026-01-18 |-8              |
|PARCEL_010|2026-05-10|2026-01-07 |123             |
|PARCEL_001|2026-01-16|2026-02-10 |-25             |
|PARCEL_023|2026-03-24|2026-01-26 |57              |
|PARCEL_020|2026-03-12|2026-02-19 |21              |
|PARCEL_013|2026-01-01|2026-02-23 |-53             |
|PARCEL_025|2026-02-04|2026-02-07 |-3              |
|PARCEL_001|2026-01-26|2026-02-10 |-15             |
|PARCEL_013|2026-01-28|2026-02-23 |-26        

In [0]:
#creating mean NDVI before sowing

from pyspark.sql.functions import avg, round

before_crop = (
    before_df
    .groupBy("crop_type")
    .agg(
        round(avg("ndvi_value"), 4)
        .alias("mean_ndvi_before")
    )
)

display(before_crop)


crop_type,mean_ndvi_before
sugarcane,0.1778
soybean,0.1705
wheat,0.1793


In [0]:
#creating mean NDVI after sowing

after_crop = (
    after_df
    .groupBy("crop_type")
    .agg(
        round(avg("ndvi_value"), 4)
        .alias("mean_ndvi_after")
    )
)

display(after_crop)

crop_type,mean_ndvi_after
sugarcane,0.3354
soybean,0.3114
wheat,0.3087


In [0]:
#calculating parcels

parcel_counts = (
    analysis_df
    .select("crop_type", "parcel_id")
    .distinct()
    .groupBy("crop_type")
    .count()
    .withColumnRenamed(
        "count",
        "n_parcels"
    )
)

display(parcel_counts)

crop_type,n_parcels
sugarcane,19
soybean,4
wheat,2


In [0]:
#final results for TASK 3

final_analysis = (
    before_crop
    .join(after_crop, "crop_type")
    .join(parcel_counts, "crop_type")
)

display(final_analysis)

crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
sugarcane,0.1778,0.3354,19
soybean,0.1705,0.3114,4
wheat,0.1793,0.3087,2


In [0]:
#validating the results

final_analysis.orderBy("crop_type").show(truncate=False)

+---------+----------------+---------------+---------+
|crop_type|mean_ndvi_before|mean_ndvi_after|n_parcels|
+---------+----------------+---------------+---------+
|soybean  |0.1705          |0.3114         |4        |
|sugarcane|0.1778          |0.3354         |19       |
|wheat    |0.1793          |0.3087         |2        |
+---------+----------------+---------------+---------+



In [0]:
pandas_df = cleaned_df.toPandas()

In [0]:
pandas_df.to_csv(
    "cleaned_parcel_timeseries.csv",
    index=False
)

In [0]:
pandas_df.to_csv(
    "/Workspace/Users/srinath.mahto@gmail.com//cleaned_parcel_timeseries.csv",
    index=False
)